# Module 16 · Họ hàng của biến đổi Fourier: nhiều chiều, Hankel, Mellin, z, Abel, Radon, Hilbert, phân số

**Nguồn sách:** Bracewell, chương 13, tr. 330–374

Notebook đi kèm bài giảng cùng số module. Chạy lần lượt từng cell; sau mỗi cell có phần **📤 Đầu ra thật** giải thích con số.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (7.5, 3.3), "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 110})


def amplitude_spectrum(x):
    """Phổ biên độ một phía: sóng hình sin biên độ A cho vạch A, thành phần một chiều cho đúng giá trị của nó"""
    a = 2*np.abs(np.fft.rfft(x))/len(x)
    a[0] /= 2
    return a


def report(key, value, fmt=".4g"):
    """In một kết quả có tên, bỏ các số 0 thừa ở cuối (trang web đọc các dòng bắt đầu bằng ▸)."""
    s = f"{value:{fmt}}"
    if "." in s and "e" not in s:
        s = s.rstrip("0").rstrip(".")
    print(f"▸ {key} = {s}")


## 1. Fourier hai và ba chiều
🎯 **Phương pháp này trả lời câu hỏi gì?** Các định lý hai chiều (xoay, cắt, affine, mômen) và biến đổi khối cầu 3D có đúng số, tính bằng tổng trên lưới và bằng công thức?

In [2]:
from scipy import integrate, special, signal
trap = getattr(np, "trapezoid", None) or np.trapz
rg = np.random.default_rng(16)
h_ = np.arange(-8, 8, 0.04); X, Y = np.meshgrid(h_, h_, indexing="ij"); dxy = 0.04
def FT2(fun, u, v): return np.sum(fun(X, Y)*np.exp(-2j*np.pi*(u*X + v*Y)))*dxy*dxy
gg = lambda x, y: np.exp(-np.pi*(x**2 + y**2))
u0, v0 = 0.5, 0.3
num = FT2(gg, u0, v0); cf = np.exp(-np.pi*(u0**2 + v0**2))
assert abs(num - cf) < 1e-10
report("g2_num", num.real, ".4f"); report("g2_cf", cf, ".4f"); report("g2_dev", max(abs(num - cf), 1e-16), ".0e")
# tích và chập
a = np.array([1, 2, 3, 2, 1.]); b = np.array([1, 3, 3, 1.])
cv = signal.convolve2d(a[None, :], b[:, None]); assert np.allclose(cv, np.outer(b, a))
report("pc_dev", max(np.max(np.abs(cv - np.outer(b, a))), 1e-16), ".0e"); report("pc_sum", cv.sum(), ".0f")
# mômen
mo_num = np.sum((X**2 + Y**2)*gg(X, Y))*dxy*dxy
Fuu = -2*np.pi; mo_der = -(Fuu + Fuu)/(4*np.pi**2)
assert abs(mo_num - 1/np.pi) < 1e-9 and abs(mo_der - 1/np.pi) < 1e-12 and abs(np.sum(gg(X, Y))*dxy*dxy - 1) < 1e-10
report("mo_F0", 1.0, ".0f"); report("mo_r2", 1/np.pi, ".4f")
# xoay
f0 = lambda x, y: np.exp(-np.pi*(4*x**2 + y**2/4)); F0f = lambda u, v: np.exp(-np.pi*(u**2/4 + 4*v**2))
th = 0.6; fr_ = lambda x, y: f0(x*np.cos(th) - y*np.sin(th), x*np.sin(th) + y*np.cos(th))
u1, v1 = 0.3, 0.2
lhs = FT2(fr_, u1, v1); rhs = F0f(u1*np.cos(th) - v1*np.sin(th), u1*np.sin(th) + v1*np.cos(th))
assert abs(lhs - rhs) < 1e-9; report("rot_dev", max(abs(lhs - rhs), 1e-16), ".0e")
# cắt
bs = 0.5; fsh = lambda x, y: gg(x + bs*y, y); u2, v2 = 0.4, 0.3
lhs = FT2(fsh, u2, v2); rhs = np.exp(-np.pi*(u2**2 + (v2 - bs*u2)**2))
assert abs(lhs - rhs) < 1e-9; report("shr_dev", max(abs(lhs - rhs), 1e-16), ".0e")
# affine
Fa = lambda x, y: np.exp(-np.pi*(x**2 + 2*y**2 + 0.5*x*y))
ap, bp, cp, dp, ep, fp = 1.2, 0.3, 0.4, -0.2, 0.9, -0.3
faf = lambda x, y: Fa(ap*x + bp*y + cp, dp*x + ep*y + fp)
u3, v3 = 0.35, -0.2; det = ap*ep - bp*dp
arg = ((ep*u3 - dp*v3)/det, (-bp*u3 + ap*v3)/det)
lhs = FT2(faf, u3, v3)
right = abs(det)**-1*np.exp(1j*2*np.pi/det*((ep*cp - bp*fp)*u3 + (ap*fp - cp*dp)*v3))*FT2(Fa, *arg)
wrong = abs(det)**-1*np.exp(-1j*2*np.pi/det*((ep*cp - bp*fp)*u3 + (ap*fp - cp*dp)*v3))*FT2(Fa, *arg)
assert abs(lhs - right) < 1e-9 and abs(lhs - wrong) > 0.1
report("aff_dev", max(abs(lhs - right), 1e-16), ".0e"); report("aff_wrong", abs(lhs - wrong), ".2f")
# 3D: khối cầu
ball = lambda s_: (np.sin(2*np.pi*s_) - 2*np.pi*s_*np.cos(2*np.pi*s_))/(2*np.pi**2*s_**3)
sv = 1.0
bnum = 4*np.pi*integrate.quad(lambda r: np.sinc(2*sv*r)*r*r, 0, 1)[0]
assert abs(bnum - ball(sv)) < 1e-12 and abs(ball(1.0) + 1/np.pi) < 1e-12
vol0 = 4*np.pi*integrate.quad(lambda r: r*r, 0, 1)[0]; assert abs(vol0 - 4*np.pi/3) < 1e-12
report("ball_s0", vol0, ".4f"); report("ball_s1", ball(1.0), ".4f"); report("ball_num", bnum, ".4f")
sg = 0.5; g3 = 4*np.pi*integrate.quad(lambda r: np.exp(-np.pi*r*r)*np.sinc(2*sg*r)*r*r, 0, 12)[0]
assert abs(g3 - np.exp(-np.pi*sg**2)) < 1e-10; report("g3_s", g3, ".4f")
cube = np.sinc(0.5)*np.sinc(0.25)*np.sinc(0.25)
xs_ = np.linspace(-0.5, 0.5, 4001); cu = trap(np.cos(2*np.pi*0.5*xs_), xs_)*trap(np.cos(2*np.pi*0.25*xs_), xs_)**2
assert abs(cu - cube) < 1e-6; report("cube_val", cube, ".4f")

▸ g2_num = 0.3436
▸ g2_cf = 0.3436
▸ g2_dev = 6e-16
▸ pc_dev = 1e-16
▸ pc_sum = 72
▸ mo_F0 = 1
▸ mo_r2 = 0.3183
▸ rot_dev = 4e-16
▸ shr_dev = 1e-15
▸ aff_dev = 2e-16
▸ aff_wrong = 0.78
▸ ball_s0 = 4.1888
▸ ball_s1 = -0.3183
▸ ball_num = -0.3183
▸ g3_s = 0.4559
▸ cube_val = 0.516


#### 📤 Đầu ra thật
Gauss 2D 0.3436 và 0.3436; chập 72; mômen 0.3183; xoay 4e-16; cắt 1e-15; affine 2e-16 và sai dấu 0.78; khối cầu 4.1888, -0.3183; Gauss 3D 0.4559; lập phương 0.516.

## 2. Hankel và hạt nhân Fourier
🎯 **Phương pháp này trả lời câu hỏi gì?** Các cặp Hankel (đĩa, mũ, Gauss) có khớp bảng, biến đổi có đối ngược, $J_{\pm1/2}$ có cho cos và sin, và Hankel $n$ chiều có cho cùng Gauss?

In [3]:
J0 = special.j0; J1 = special.j1
hank = lambda fun, q, upper=np.inf: 2*np.pi*integrate.quad(lambda r: fun(r)*J0(2*np.pi*q*r)*r, 0, upper, limit=400)[0]
a_d, q_ = 1.5, 0.4
disk_num = hank(lambda r: 1.0, q_, a_d); disk_cf = a_d*J1(2*np.pi*a_d*q_)/q_
assert abs(disk_num - disk_cf) < 1e-10; report("hk_num", disk_num, ".4f"); report("hk_cf", disk_cf, ".4f")
ae = 1.3; e_num = hank(lambda r: np.exp(-ae*r), q_, 60); e_cf = 2*np.pi*ae*(ae**2 + 4*np.pi**2*q_**2)**-1.5
assert abs(e_num - e_cf) < 1e-9; report("hk_e_num", e_num, ".4f"); report("hk_e_cf", e_cf, ".4f")
gq = hank(lambda r: np.exp(-np.pi*r*r), 0.5, 12); assert abs(gq - np.exp(-np.pi*0.25)) < 1e-10; report("hk_g", gq, ".4f")
# đối ngược: áp dụng hai lần cho e^{-ar}
Fq = lambda q: 2*np.pi*ae*(ae**2 + 4*np.pi**2*q**2)**-1.5
back = 2*np.pi*integrate.quad(lambda q: Fq(q)*J0(2*np.pi*q*0.5)*q, 0, 400, limit=2000)[0]
tail = 2*np.pi*(2*np.pi*ae)*(2*np.pi)**-3*(1/400)  # đuôi ~ 1/q² của tích phân nhân J0 ≈ dao động, bỏ qua
assert abs(back - np.exp(-ae*0.5)) < 5e-4
report("hk_rec_f", np.exp(-ae*0.5), ".4f"); report("hk_rec_dev", max(abs(back - np.exp(-ae*0.5)), 1e-16), ".0e")
# định lý
hd_int = 2*np.pi*integrate.quad(lambda r: np.exp(-np.pi*r*r)*r, 0, 12)[0]; assert abs(hd_int - 1) < 1e-10
rl = 2*np.pi*integrate.quad(lambda r: np.exp(-2*np.pi*r*r)*r, 0, 12)[0]; rr = 2*np.pi*integrate.quad(lambda q: np.exp(-2*np.pi*q*q)*q, 0, 12)[0]
assert abs(rl - rr) < 1e-12; report("hd_int", hd_int, ".0f"); report("hd_ray_l", rl, ".4f"); report("hd_ray_r", rr, ".4f")
# Abel rồi Fourier: Hankel = Fourier ∘ Abel
xg = np.arange(0, 6.0001, 0.02)
A_num = np.array([2*integrate.quad(lambda t: np.exp(-np.pi*(x_**2 + t**2)), 0, 8)[0] for x_ in xg])
assert np.max(np.abs(A_num - np.exp(-np.pi*xg**2))) < 1e-9
sgg = np.arange(0, 6.0001, 0.02)
Fs_ = np.array([2*trap(A_num*np.cos(2*np.pi*s_*xg), xg) for s_ in sgg])
q_half = 0.5; nh_abel = 2*np.pi*trap(Fs_*J0(2*np.pi*sgg*0.0 + 0)*0, sgg) if False else 0
Fh_q = np.interp(q_half, sgg, Fs_)
nh_direct = hank(lambda r: np.exp(-np.pi*r*r), 0.5, 12)
assert abs(Fh_q - np.exp(-np.pi*0.25)) < 1e-4 and abs(nh_direct - np.exp(-np.pi*0.25)) < 1e-10
report("nh_direct", nh_direct, ".4f"); report("nh_abel", Fh_q, ".4f"); report("nh_cf", np.exp(-np.pi*0.25), ".4f")
# J_{±1/2}
z = 2.3; jm = special.jv(-0.5, z) - np.sqrt(2/(np.pi*z))*np.cos(z); jp = special.jv(0.5, z) - np.sqrt(2/(np.pi*z))*np.sin(z)
assert abs(jm) < 1e-13 and abs(jp) < 1e-13; report("jv_dev", max(abs(jm), abs(jp), 1e-16), ".0e")
# Hankel n chiều
def hn(n, q):
    return 2*np.pi*q**(1 - n/2)*integrate.quad(lambda r: np.exp(-np.pi*r*r)*special.jv(n/2 - 1, 2*np.pi*q*r)*r**(n/2), 0, 12)[0]
vals = [hn(n, 0.5) for n in (1, 2, 3)]
assert max(abs(v_ - np.exp(-np.pi*0.25)) for v_ in vals) < 1e-9
report("nd_val", vals[0], ".4f"); report("nd_dev", max(max(abs(v_ - np.exp(-np.pi*0.25)) for v_ in vals), 1e-16), ".0e")
zj = special.jn_zeros(1, 1)[0]; assert abs(zj - 3.8317) < 1e-4; report("airy_z", zj, ".4f")

▸ hk_num = 0.094
▸ hk_cf = 0.094
▸ hk_e_num = 0.3605
▸ hk_e_cf = 0.3605
▸ hk_g = 0.4559
▸ hk_rec_f = 0.522
▸ hk_rec_dev = 7e-09
▸ hd_int = 1
▸ hd_ray_l = 0.5
▸ hd_ray_r = 0.5
▸ nh_direct = 0.4559
▸ nh_abel = 0.4559
▸ nh_cf = 0.4559
▸ jv_dev = 5e-16
▸ nd_val = 0.4559
▸ nd_dev = 3e-16
▸ airy_z = 3.8317


#### 📤 Đầu ra thật
Đĩa 0.094 và 0.094; mũ 0.3605 và 0.3605; Gauss 0.4559; đối ngược 0.522 lệch 7e-09; Rayleigh 0.5, 0.5; Hankel 0.4559, 0.4559; $J_{1/2}$ 5e-16; $n$ chiều 0.4559, 3e-16; nghiệm jinc 3.8317.

## 3. Mellin và z
🎯 **Phương pháp này trả lời câu hỏi gì?** Các cặp Mellin (mũ, sin, phân thức), điều kiện hạt nhân Fourier, và các cặp, định lý của biến đổi z có đúng số?

In [4]:
G = special.gamma
s_ = 2.5
ml_direct = integrate.quad(lambda x: np.exp(-x)*x**(s_ - 1), 0, np.inf)[0]
ml_lap = integrate.quad(lambda t: np.exp(-np.exp(-t))*np.exp(-s_*t), -30, 40, limit=400)[0]
assert abs(ml_direct - G(s_)) < 1e-9 and abs(ml_lap - G(s_)) < 1e-8
report("ml_gam", G(s_), ".4f"); report("ml_dev", max(abs(ml_lap - ml_direct), 1e-16), ".0e")
def mellin_sin(s):
    a = integrate.quad(lambda x: np.sin(x), 0, 1, weight="alg", wvar=(s - 1, 0))[0]
    b = integrate.quad(lambda x: x**(s - 1), 1, np.inf, weight="sin", wvar=1)[0]
    return a + b
assert abs(mellin_sin(0.5) - G(0.5)*np.sin(np.pi/4)) < 1e-8; report("ml_sin", mellin_sin(0.5), ".4f")
rat = integrate.quad(lambda x: x**(0.3 - 1)/(1 + x), 0, np.inf)[0]; assert abs(rat - np.pi/np.sin(0.3*np.pi)) < 1e-6
report("ml_rat", np.pi/np.sin(0.3*np.pi), ".4f")
h2 = integrate.quad(lambda x: x**(2 - 1), 0, 1)[0]; assert abs(h2 - 0.5) < 1e-12; report("ml_h2", h2, ".1f")
# mômen
FM = lambda s: integrate.quad(lambda x: x**(s - 1), 0, 1)[0]
assert abs(FM(1) - 1) < 1e-12 and abs(FM(2) - 0.5) < 1e-12 and abs(FM(3) - 1/3) < 1e-12
cx = FM(2)/FM(1); rgy = np.sqrt(FM(3)/FM(1)); var = FM(3)/FM(1) - cx**2
assert abs(var - 1/12) < 1e-12
report("mm_c", cx, ".1f"); report("mm_rg", rgy, ".4f"); report("mm_v", var, ".4f")
sc = integrate.quad(lambda x: np.exp(-2*x)*x**1.5, 0, np.inf)[0]; assert abs(sc - 2**-2.5*G(2.5)) < 1e-9
report("mm_sc", sc, ".4f")
# hạt nhân Fourier
def KM(s):
    a = integrate.quad(lambda x: np.cos(2*np.pi*x), 0, 1, weight="alg", wvar=(s - 1, 0))[0]
    b = integrate.quad(lambda x: x**(s - 1), 1, np.inf, weight="cos", wvar=2*np.pi)[0]
    return 2*(a + b)
Kcf = lambda s: 2*(2*np.pi)**(-s)*G(s)*np.cos(np.pi*s/2)
assert abs(KM(0.3) - Kcf(0.3)) < 1e-8 and abs(Kcf(0.3)*Kcf(0.7) - 1) < 1e-12 and abs(KM(0.3)*KM(0.7) - 1) < 1e-7
report("fk_k", Kcf(0.3), ".4f"); report("fk_num", KM(0.3), ".4f"); report("fk_prod", Kcf(0.3)*Kcf(0.7), ".4f")
# biến đổi z
fz = np.array([3, 1, 4, 1, 5, 9, 2, 6.]); zf = lambda zz: np.sum(fz*zz**(-np.arange(len(fz))))
p_ = 0.7; lap = np.sum(fz*np.exp(-np.arange(8)*p_)); assert abs(lap - zf(np.exp(p_))) < 1e-12
report("z_lap", lap, ".4f"); report("z_lap_dev", max(abs(lap - zf(np.exp(p_))), 1e-16), ".0e")
Nz = len(fz); zk = np.exp(2j*np.pi*np.arange(Nz)/Nz); Fz = np.array([zf(z_) for z_ in zk])
assert np.allclose(Fz, np.fft.fft(fz)); report("z_dft", max(np.max(np.abs(Fz - np.fft.fft(fz))), 1e-16), ".0e")
out = np.convolve([2, 1], [8, 4, 2, 1]); assert list(out) == [16, 16, 8, 4, 1]
qd, rd = np.polydiv(out.astype(float), np.array([2., 1.])); assert np.allclose(qd, [8, 4, 2, 1]) and np.allclose(rd, 0)
report("z_o1", out[0], "d"); report("z_o5", out[-1], "d"); report("z_rem", np.max(np.abs(rd)) + 0.0, ".0f")
n_ = np.arange(300); zz = 2.0
S1 = np.sum(n_*zz**-n_); S2 = np.sum(n_**2*zz**-n_.astype(float)); al = 0.7; Sc = np.sum(np.cos(al*n_)*zz**-n_.astype(float))
zi = 1/zz
assert abs(S1 - zi/(1 - zi)**2) < 1e-10 and abs(S2 - zi*(1 + zi)/(1 - zi)**3) < 1e-10
assert abs(Sc - (1 - zi*np.cos(al))/(1 - 2*zi*np.cos(al) + zi**2)) < 1e-10
report("z_n1", S1, ".0f"); report("z_n2", S2, ".0f"); report("z_cos", Sc, ".4f")
Fpoly = lambda z_: np.sum(fz*z_**(-np.arange(8)))
zt = 1.7; hh = 1e-6
der = (Fpoly(zt + hh) - Fpoly(zt - hh))/(2*hh)
nf = np.sum(np.arange(8)*fz*zt**(-np.arange(8)))
assert abs(nf - (-zt*der)) < 1e-6; report("z_dev", max(abs(nf + zt*der), 1e-16), ".0e")

▸ ml_gam = 1.3293
▸ ml_dev = 3e-12
▸ ml_sin = 1.2533
▸ ml_rat = 3.8832
▸ ml_h2 = 0.5
▸ mm_c = 0.5
▸ mm_rg = 0.5774
▸ mm_v = 0.0833
▸ mm_sc = 0.235
▸ fk_k = 3.0715
▸ fk_num = 3.0715
▸ fk_prod = 1
▸ z_lap = 5.2559
▸ z_lap_dev = 9e-16
▸ z_dft = 1e-14
▸ z_o1 = 16
▸ z_o5 = 1
▸ z_rem = 0
▸ z_n1 = 2
▸ z_n2 = 6
▸ z_cos = 1.2729
▸ z_dev = 2e-09


#### 📤 Đầu ra thật
$\Gamma(2.5)$ = 1.3293; $\sin$ 1.2533; $1/(1+x)$ 3.8832; trọng tâm 0.5, phương sai 0.0833; hạt nhân Fourier 3.0715, tích 1; z: 5.2559, 1e-14; đầu ra 16…1; chuỗi 2, 6, 1.2729.

## 4. Abel, Radon, chiếu ngược
🎯 **Phương pháp này trả lời câu hỏi gì?** Biến đổi Abel số có khớp bảng của sách, định lý lát chiếu có đúng, và chiếu ngược có sửa đổi có tái tạo được mật độ từ các hình chiếu không?

In [5]:
fA = lambda fun, x, upper=np.inf: 2*integrate.quad(lambda t: fun(np.sqrt(x*x + t*t)), 0, upper, limit=200)[0]
disk = lambda r: 1.0 if r < 1 else 0.0
adisk = 2*integrate.quad(lambda t: 1.0, 0, np.sqrt(1 - 0.36))[0]; assert abs(adisk - 1.6) < 1e-12
report("ab_disk", adisk, ".1f")
report("ab_gauss", fA(lambda r: np.exp(-np.pi*r*r), 0.5, 10), ".4f")
assert abs(fA(lambda r: np.exp(-np.pi*r*r), 0.5, 10) - np.exp(-np.pi*0.25)) < 1e-10
f1r = lambda r: max(1 - r, 0.0)
def fA1(x):
    if x >= 1: return 0.0
    return 2*integrate.quad(lambda t: 1 - np.sqrt(x*x + t*t), 0, np.sqrt(1 - x*x))[0]
book = [1, .9651, .8881, .7853, .6658, .5368, .4045, .2753, .1564, .0575]
vals = [fA1(x/10) for x in range(10)]
assert max(abs(v_ - b_) for v_, b_ in zip(vals, book)) < 1.5e-3
report("ab_x1", vals[1], ".4f"); report("ab_x5", vals[5], ".4f"); report("ab_x9", vals[9], ".4f"); report("ab_0", vals[0], ".0f")
area = integrate.quad(fA1, -1, 1, points=[0])[0]; assert abs(area - np.pi/3) < 1e-8; report("ab_area", area, ".4f")
# bảng hệ số
cn = lambda n: 2*(np.sqrt(n + 1) - np.sqrt(n))
assert abs(cn(0) - 2) < 1e-12; report("ab_k1", cn(1), ".3f"); report("ab_k2", cn(2), ".3f"); report("ab_k3", cn(3), ".3f")
alg = sum(cn(n)*np.sqrt(10 - (5 + n + 0.5)) for n in range(5)); exact = np.pi/2*(10 - 5)
assert abs(alg - 7.78) < 0.02 and abs(exact - 7.854) < 1e-3
ex_num = integrate.quad(lambda r: np.sqrt(10 - r)/np.sqrt(r - 5), 5, 10)[0]; assert abs(ex_num - exact) < 1e-6
report("ab_alg", alg, ".2f"); report("ab_exact", exact, ".3f")
# vòng Abel-Fourier-Hankel
xg = np.arange(0, 6.0001, 0.02)
A_num = np.array([fA(lambda r: np.exp(-np.pi*r*r), x_, 8) for x_ in xg])
sgg = np.arange(0, 6.0001, 0.02)
Fs_ = np.array([2*trap(A_num*np.cos(2*np.pi*s_*xg), xg) for s_ in sgg])
r0 = 0.6; back = 2*np.pi*integrate.simpson(Fs_*special.j0(2*np.pi*sgg*r0)*sgg, x=sgg)
assert abs(back - np.exp(-np.pi*r0**2)) < 1e-6
report("arh_val", np.exp(-np.pi*r0**2), ".4f"); report("arh_dev", max(abs(back - np.exp(-np.pi*r0**2)), 1e-16), ".0e")
# Radon và định lý lát chiếu
aa, bb = 2.0, 0.5; ell = lambda x, y: np.exp(-np.pi*(x**2/aa**2 + y**2/bb**2))
th = 0.6; kap = aa**2*np.cos(th)**2 + bb**2*np.sin(th)**2
proj_cf = lambda R: aa*bb/np.sqrt(kap)*np.exp(-np.pi*R**2/kap)
proj_num = lambda R: integrate.quad(lambda t: ell(R*np.cos(th) - t*np.sin(th), R*np.sin(th) + t*np.cos(th)), -30, 30, limit=200)[0]
assert abs(proj_num(0.3) - proj_cf(0.3)) < 1e-9; report("rd_num", proj_num(0.3), ".4f"); report("rd_cf", proj_cf(0.3), ".4f")
Rg = np.linspace(-12, 12, 4801); gR = np.array([proj_cf(R) for R in Rg]); q0 = 0.4
one_d = trap(gR*np.exp(-2j*np.pi*q0*Rg), Rg); slice_ = aa*bb*np.exp(-np.pi*q0**2*kap)
assert abs(one_d - slice_) < 1e-9; report("ps_dev", max(abs(one_d - slice_), 1e-16), ".0e")
# nhân sửa đổi
M = 6.0; Rk = np.arange(-600, 600, 0.02)
kern = 2*M*np.sinc(2*M*Rk) - M*np.sinc(M*Rk)**2
for q, want in ((2.0, 2.0/M), (8.0, 0.0)):
    got = trap(kern*np.cos(2*np.pi*q*Rk), Rk); assert abs(got - want) < 2e-3
report("kf_2", trap(kern*np.cos(2*np.pi*2.0*Rk), Rk), ".3f"); report("kf_8", round(trap(kern*np.cos(2*np.pi*8.0*Rk), Rk), 3) + 0.0, ".0f")
# chiếu ngược có sửa đổi
dR = 1/(4*M); Rgrid = np.arange(-8, 8, dR); nR = len(Rgrid)
def proj_two(theta, R):
    out_ = 0
    for A_, w_, cx_, cy_ in ((1.0, 0.5, 0.0, 0.0), (0.6, 0.3, 0.7, -0.5)):
        cn_ = cx_*np.cos(theta) + cy_*np.sin(theta)
        out_ = out_ + A_*w_*np.exp(-np.pi*(R - cn_)**2/w_**2)
    return out_
qf = np.fft.fftfreq(nR, dR); ramp = np.where(np.abs(qf) < M, np.abs(qf)/M, 0)
f_true = lambda x, y: np.exp(-np.pi*(x**2 + y**2)/0.25) + 0.6*np.exp(-np.pi*((x - 0.7)**2 + (y + 0.5)**2)/0.09)
pts = [(0, 0), (0.7, -0.5), (0.3, 0.2), (-0.5, 0.4)]
def recon(nth):
    thetas = np.linspace(0, np.pi, nth, endpoint=False)
    mods = [np.real(np.fft.ifft(np.fft.fft(proj_two(t_, Rgrid))*ramp)) for t_ in thetas]
    return [M*sum(np.interp(x_*np.cos(t_) + y_*np.sin(t_), Rgrid, g0) for t_, g0 in zip(thetas, mods))*np.pi/nth for (x_, y_) in pts]
r180 = recon(180); r90 = recon(90)
errs = [abs(r - f_true(x_, y_)) for r, (x_, y_) in zip(r180, pts)]
assert max(errs) < 0.02 and abs(r180[0] - 1) < 0.01 and max(abs(np.array(r180) - np.array(r90))) < 1e-3
report("fb_c", r180[0], ".4f"); report("fb_dev", max(errs), ".3f")

▸ ab_disk = 1.6
▸ ab_gauss = 0.4559
▸ ab_x1 = 0.9651
▸ ab_x5 = 0.5368
▸ ab_x9 = 0.0575
▸ ab_0 = 1
▸ ab_area = 1.0472
▸ ab_k1 = 0.828
▸ ab_k2 = 0.636
▸ ab_k3 = 0.536
▸ ab_alg = 7.79
▸ ab_exact = 7.854
▸ arh_val = 0.3227
▸ arh_dev = 2e-07
▸ rd_num = 0.5399
▸ rd_cf = 0.5399
▸ ps_dev = 1e-16
▸ kf_2 = 0.333
▸ kf_8 = 0
▸ fb_c = 0.9994
▸ fb_dev = 0.013


#### 📤 Đầu ra thật
Abel: đĩa 1.6, Gauss 0.4559; $f=1-r$: 0.9651, 0.5368, 0.0575, diện tích 1.0472; hệ số 0.828, 0.636, 0.536; thuật toán 7.79 so với 7.854; vòng 0.3227 lệch 2e-07; Radon 0.5399, 0.5399, lát cắt 1e-16; nhân 0.333, 0; tái tạo 0.9994, sai 0.013.

## 5. Hilbert và Fourier phân số
🎯 **Phương pháp này trả lời câu hỏi gì?** Hilbert dời pha $\pm\pi/2$ có đúng, tín hiệu giải tích có khớp thư viện, đường bao và tần số tức thời có đúng, cặp nhân quả có đúng, và biến đổi Fourier phân số có cộng tính và hàm riêng Hermite-Gauss không?

In [6]:
# Hilbert bằng FFT (bộ lọc i sgn s)
def hilb(x):
    Xf = np.fft.fft(x); sgf = np.sign(np.fft.fftfreq(len(x))); return np.real(np.fft.ifft(Xf*1j*sgf))
Np = 1024; tp = np.arange(Np)/Np
c5 = np.cos(2*np.pi*5*tp); s5 = np.sin(2*np.pi*5*tp)
assert np.max(np.abs(hilb(c5) + s5)) < 1e-12 and np.max(np.abs(hilb(s5) - c5)) < 1e-12
report("hb_cos", max(np.max(np.abs(hilb(c5) + s5)), 1e-16), ".0e")
xr = rg.standard_normal(Np); Xr = np.fft.fft(xr); Xr[0] = 0; Xr[Np//2] = 0; xr = np.real(np.fft.ifft(Xr))
assert np.max(np.abs(hilb(hilb(xr)) + xr)) < 1e-12; report("hb_two", max(np.max(np.abs(hilb(hilb(xr)) + xr)), 1e-16), ".0e")
# Hilbert của Π(x)
def hilb_rect(x0):
    if abs(abs(x0) - 0.5) < 1e-12: return np.nan
    if abs(x0) > 0.5: return (1/np.pi)*integrate.quad(lambda xp: 1/(xp - x0), -0.5, 0.5)[0]
    return (1/np.pi)*integrate.quad(lambda xp: 1.0, -0.5, 0.5, weight="cauchy", wvar=x0)[0]
cf = lambda x0: (1/np.pi)*np.log(abs((x0 - 0.5)/(x0 + 0.5)))
assert abs(hilb_rect(1.0) - cf(1.0)) < 1e-10 and abs(hilb_rect(0.2) - cf(0.2)) < 1e-10
report("hr_1", cf(1.0), ".4f"); report("hr_02", cf(0.2), ".4f")
# tín hiệu giải tích
b_ = signal.firwin(101, [0.10, 0.20], pass_zero=False); nb = signal.lfilter(b_, 1, rg.standard_normal(4096))[200:200 + 2048]
an_scipy = signal.hilbert(nb); an_ours = nb - 1j*hilb(nb)
assert np.max(np.abs(an_scipy - an_ours)) < 1e-10
Fan = np.fft.fft(an_ours); neg = np.max(np.abs(Fan[len(nb)//2 + 1:]))/np.max(np.abs(Fan))
assert neg < 1e-10
report("an_dev", max(np.max(np.abs(an_scipy - an_ours)), 1e-16), ".0e"); report("an_neg", max(neg, 1e-16), ".0e")
# đường bao, tần số tức thời
Ns = 4096; ts = np.arange(Ns)*2*np.pi/Ns*4   # 4 chu kỳ của Ω = 3? dùng tần số nguyên
tt = np.arange(Ns)/Ns*2*np.pi
am = (1 + 0.5*np.cos(3*tt))*np.cos(50*tt); env = np.abs(signal.hilbert(am))
assert np.max(np.abs(env - (1 + 0.5*np.cos(3*tt)))) < 1e-9
report("env_dev", max(np.max(np.abs(env - (1 + 0.5*np.cos(3*tt)))), 1e-16), ".0e")
fm = np.cos(50*tt + 2*np.sin(3*tt)); ph = np.unwrap(np.angle(signal.hilbert(fm)))
inst = np.gradient(ph, tt); dev = np.max(inst[50:-50]) - 50
assert abs(dev - 6) < 0.05; report("fm_dev", dev, ".3f")
# Kramers-Kronig
Gk = lambda u: 1/(1 + 4*np.pi**2*u**2); f0 = 0.3
pv1 = integrate.quad(lambda u: Gk(u), f0 - 1, f0 + 1, weight="cauchy", wvar=f0)[0]
rest = integrate.quad(lambda u: Gk(u)/(u - f0), f0 + 1, np.inf)[0] + integrate.quad(lambda u: Gk(u)/(u - f0), -np.inf, f0 - 1)[0]
Bnum = (pv1 + rest)/np.pi; Bcf = -2*np.pi*f0/(1 + 4*np.pi**2*f0**2)
assert abs(Bnum - Bcf) < 1e-9
report("kk_b", Bcf, ".4f"); report("kk_num", Bnum, ".4f"); report("kk_dev", max(abs(Bnum - Bcf), 1e-16), ".0e")
# Hilbert rời rạc: 11 hệ số
nn = np.arange(-5, 6); hx = np.where(nn == 0, 0, -1/(np.pi*np.where(nn == 0, 1, nn)))
assert np.allclose(np.abs(hx[[0, 1, 2, 3, 4]]), [.064, .080, .106, .159, .318], atol=6e-4)
report("dh_a", abs(hx[0]), ".3f"); report("dh_b", abs(hx[4]), ".3f")
sv = np.linspace(0.01, 0.49, 49); Hs_ = np.array([np.sum(hx*np.exp(-2j*np.pi*s_*nn)) for s_ in sv])
assert np.all(np.abs(np.angle(Hs_) - np.pi/2) < 1e-9)
amp = np.abs(Hs_); ip = int(np.argmax(amp))
report("dh_s", sv[ip], ".2f"); report("dh_peak", amp[ip], ".3f"); report("dh_a10", amp[9], ".3f"); report("dh_a25", amp[24], ".3f"); report("dh_a40", amp[39], ".3f")
# Hilbert bằng DHT
def dht(f):
    f = np.asarray(f, float); N = len(f); t = np.arange(N)
    return np.array([np.sum(f*(np.cos(2*np.pi*v*t/N) + np.sin(2*np.pi*v*t/N))) for v in range(N)])/N
def idht(H):
    N = len(H); v = np.arange(N); return np.array([np.sum(H*(np.cos(2*np.pi*v*t/N) + np.sin(2*np.pi*v*t/N))) for t in range(N)])
f8 = rg.standard_normal(8); F8 = np.fft.fft(f8); F8[0] = 0; F8[4] = 0; f8 = np.real(np.fft.ifft(F8))
H8 = dht(f8); Hsw = np.array([H8[0], -H8[7], -H8[6], -H8[5], H8[4], H8[3], H8[2], H8[1]])
hd = idht(Hsw); ref = hilb(f8)
assert np.max(np.abs(hd - ref)) < 1e-12 and np.max(np.abs(hd + ref)) > 0.1
report("hh_dev", max(np.max(np.abs(hd - ref)), 1e-16), ".0e"); report("hh_wrong", np.max(np.abs(hd + ref)), ".2f")
# Fourier phân số
tg = np.arange(-7, 7.0001, 0.02); dtg = 0.02
def frft_mat(alpha):
    cot = 1/np.tan(alpha); sn = np.sin(alpha)
    return np.sqrt(1 - 1j*cot)*np.exp(1j*np.pi*cot*(tg[:, None]**2 + tg[None, :]**2))*np.exp(-2j*np.pi*tg[:, None]*tg[None, :]/sn)*dtg
herm = lambda n: special.eval_hermite(n, np.sqrt(2*np.pi)*tg)*np.exp(-np.pi*tg**2)
al7 = 0.7; K7 = frft_mat(al7)
eig = max(np.max(np.abs(K7 @ herm(n) - np.exp(-1j*n*al7)*herm(n))) for n in range(4))
assert eig < 1e-10; report("fr_eig", max(eig, 1e-16), ".0e")
gtest = np.exp(-np.pi*(tg - 0.6)**2)*(1 + 0.3*tg)
addd = np.max(np.abs(frft_mat(0.4) @ (frft_mat(0.5) @ gtest) - frft_mat(0.9) @ gtest)); assert addd < 1e-9
report("fr_add", max(addd, 1e-16), ".0e")
Kf = frft_mat(np.pi/2 - 1e-12); gsh = np.exp(-np.pi*(tg - 0.6)**2)
ft_true = np.exp(-np.pi*tg**2)*np.exp(-1j*np.pi*0.6*2*tg)*0 + np.sum(gsh[None, :]*np.exp(-2j*np.pi*tg[:, None]*tg[None, :]), axis=1)*dtg
assert np.max(np.abs(Kf @ gsh - ft_true)) < 1e-9; report("fr_ft", max(np.max(np.abs(Kf @ gsh - ft_true)), 1e-16), ".0e")
K5 = frft_mat(0.5*np.pi/2); out5 = K5 @ herm(1); ph5 = np.angle(np.sum(out5*herm(1))/np.sum(herm(1)**2))
assert abs(ph5 + np.pi/4) < 1e-6; report("fr_ph", ph5, ".4f")

▸ hb_cos = 3e-15
▸ hb_two = 1e-15
▸ hr_1 = -0.3497
▸ hr_02 = -0.2697
▸ an_dev = 4e-16
▸ an_neg = 1e-16
▸ env_dev = 8e-14
▸ fm_dev = 6
▸ kk_b = -0.414
▸ kk_num = -0.414
▸ kk_dev = 1e-16
▸ dh_a = 0.064
▸ dh_b = 0.318
▸ dh_s = 0.08
▸ dh_peak = 1.006
▸ dh_a10 = 0.972
▸ dh_a25 = 0.552
▸ dh_a40 = 0.18
▸ hh_dev = 1e-15
▸ hh_wrong = 2.21


▸ fr_eig = 1e-13


▸ fr_add = 3e-14


▸ fr_ft = 1e-12


▸ fr_ph = -0.7854


#### 📤 Đầu ra thật
Hilbert: 3e-15, 1e-15; $\Pi$: -0.3497, -0.2697; giải tích 4e-16, 1e-16; đường bao 8e-14; FM 6; Kramers-Kronig -0.414, -0.414; 11 hệ số: 0.318, cực đại 1.006 tại 0.08; DHT 1e-15; phân số: 1e-13, 3e-14, 1e-12, pha -0.7854.